# ReVoice — Voice Cloning API (XTTS-v2 + F5-TTS + IndicF5)

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save

Run cells **top to bottom, once each**. Don't run any old cells from previous notebooks.

Licence note: F5-TTS model weights are released for **non-commercial use** (fine for a college project).

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Install everything (3–5 minutes)
Installs XTTS, then F5-TTS **without letting it change other packages**, then ONE final line pins the known-working versions, then a conflict check (diffusers / gradio warnings can be ignored: ReVoice doesn't use them).
**After this finishes: Runtime → Restart session**, then continue from Step 2b.

In [ ]:
# XTTS + server tools
!pip install -q coqui-tts fastapi uvicorn pyngrok soundfile

# F5-TTS: --no-deps so it can't overwrite transformers/numpy
!pip install -q --no-deps f5-tts cached_path
!pip install -q ema_pytorch torchdiffeq vocos x_transformers hydra-core omegaconf jieba pypinyin unidecode pydub wandb
!pip install -q rjieba || echo "rjieba optional - skipped"

# SpeechBrain (metrics only): --no-deps so it can't change transformers/numpy/tokenizers/huggingface-hub.
!pip install -q --no-deps speechbrain hyperpyyaml "ruamel.yaml>=0.17.28,<0.19" ruamel.yaml.clib

# ffmpeg converts the browser's webm/opus recordings
!apt-get -qq install -y ffmpeg > /dev/null

# ONE final pinning line with the known-working versions; it must stay LAST so nothing changes them after.
!pip install -q tomli transformers_stream_generator boto3 bitsandbytes "transformers==4.57.6" "huggingface-hub>=0.34.0,<1.0" "tokenizers>=0.22.0,<=0.23.0" "numpy<2.3"

# Conflict check for the packages ReVoice uses (diffusers/gradio warnings are expected and ignored).
!pip check 2>&1 | grep -iE "transformers|coqui|tts|f5|librosa|numba|torch|cached|speechbrain|hyperpyyaml" | grep -viE "^(diffusers|gradio)" || echo "No conflicts in the packages ReVoice uses."

print("Install finished. Now: Runtime -> Restart session, then run Step 2b.")

## Step 2b — Check versions (after restarting)
If this errors, don't continue.

In [ ]:
import transformers, numpy, librosa, tokenizers, huggingface_hub
from packaging.version import Version
from transformers.pytorch_utils import isin_mps_friendly
from transformers.utils.import_utils import is_torch_greater_or_equal
print("transformers   ", transformers.__version__)
print("tokenizers     ", tokenizers.__version__)
print("huggingface-hub", huggingface_hub.__version__)
print("numpy          ", numpy.__version__)

# compare with the known-working set from Step 2
problems = []
if transformers.__version__ != "4.57.6":
    problems.append(f"transformers is {transformers.__version__}, expected 4.57.6")
if not Version("0.22.0") <= Version(tokenizers.__version__) <= Version("0.23.0"):
    problems.append(f"tokenizers is {tokenizers.__version__}, expected 0.22.0 to 0.23.0")
if not Version("0.34.0") <= Version(huggingface_hub.__version__) < Version("1.0"):
    problems.append(f"huggingface-hub is {huggingface_hub.__version__}, expected 0.34 up to (not incl.) 1.0")
if Version(numpy.__version__) >= Version("2.3"):
    problems.append(f"numpy is {numpy.__version__}, expected below 2.3")

import f5_tts.api
print("f5-tts importable")
import speechbrain
print("speechbrain    ", speechbrain.__version__, "(for the voice metrics)")
if problems:
    raise RuntimeError("Wrong versions:\n - " + "\n - ".join(problems) +
                       "\nRe-run Step 2, then Runtime -> Restart session.")
print("All checks passed")

## Step 3 — Load XTTS-v2 (~2 GB download first time)

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
xtts_model = tts.synthesizer.tts_model
SAMPLE_RATE = 24000
print("XTTS-v2 loaded on", device)

## Step 3b — Load F5-TTS (~1.5 GB download first time)
Also warms up Whisper (the speech-to-text model F5 uses for the reference transcript), so it downloads now and not during your first request.
If this fails, XTTS still works — the app will report the F5 error when you pick it.

In [ ]:
f5 = None
try:
    from f5_tts.api import F5TTS
    f5 = F5TTS(device=device)
    print("F5-TTS loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nF5-TTS failed to load (XTTS still available):", e)

if f5 is not None:
    # Warm-up downloads Whisper (~1.5 GB) now, so the first real request doesn't time out.
    try:
        from importlib.resources import files
        example = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
        print("Whisper warm-up transcript:", f5.transcribe(example))
    except Exception as e:
        print("Whisper warm-up failed (F5 will try again on the first request):", e)

## Step 3c — Load the evaluation models (small, ~100 MB)
Used only to MEASURE results: speaker similarity (SpeechBrain ECAPA-TDNN) and an English accent estimate (SpeechBrain CommonAccent). If they fail to load, speech still works and those numbers show as n/a.

In [ ]:
# SpeechBrain models used only to measure results: speaker similarity and accent (CommonAccent).
# If one fails to load, speech still works and that metric shows n/a.
spk_model = None
accent_model = None

try:
    from speechbrain.inference.speaker import SpeakerRecognition
    spk_model = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/content/metric_models/spkrec-ecapa-voxceleb",
        run_opts={"device": device})
    print("Speaker similarity model loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("Speaker similarity model failed to load (similarity will be n/a):", e)

try:
    from speechbrain.inference.classifiers import EncoderClassifier
    accent_model = EncoderClassifier.from_hparams(
        source="Jzuluaga/accent-id-commonaccent_ecapa",
        savedir="/content/metric_models/accent-id-commonaccent_ecapa",
        run_opts={"device": device})
    print("Accent model loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("Accent model failed to load (accent will be n/a):", e)

## Step 3d — IndicF5: 11 Indian languages (optional, ~1.4 GB download)
Speaks **Assamese, Bengali, Gujarati, Hindi, Kannada, Malayalam, Marathi, Odia, Punjabi, Tamil and Telugu**
in the user's own voice, even from an **English** recording (cross-lingual). This cell only downloads it;
it loads on the GPU with the first Indian-language request (~20 s once), then each sentence takes ~15–20 s.

One-time setup (IndicF5 is gated):
1. Open https://huggingface.co/ai4bharat/IndicF5 while logged in and **accept the terms**.
2. https://huggingface.co/settings/tokens → **Create new token** → type **Read** → copy it.
3. Colab **key icon** (left bar) → **Add new secret** → name `HF_TOKEN`, value = the token, **Notebook access** on.

If this cell fails, XTTS and F5 still work; only "IndicF5" requests reply with the reason.
Settings come from `indicf5_test.ipynb` Step T7: explicit duration from a syllable count, the English
Whisper transcript of the reference, 32 steps, peak-normalised to -1 dBFS.
**Licence:** IndicF5 weights are for non-commercial use (like F5-TTS).

In [ ]:
# IndicF5 (11 Indian languages). This cell only DOWNLOADS it; it loads on the GPU with the
# first Indian-language request. The settings are the ones indicf5_test.ipynb Step T7 chose.
import os, threading, time
import numpy as np, soundfile as sf
from google.colab import userdata
from huggingface_hub import login, hf_hub_download, HfApi

INDICF5_REPO = "ai4bharat/IndicF5"
INDICF5_NFE_STEPS = 32          # T7: CER 0.00 at 32 steps; 64 was slower for no gain
INDICF5_PEAK_DBFS = -1.0        # output scaled so its loudest point is -1 dBFS (no clipping)
indicf5 = None                  # (model, vocoder) once loaded
indicf5_files = None            # (weights, vocab) on disk after this cell
indicf5_error = None            # why IndicF5 can't be used (shown in the API's reply)
indicf5_lock = threading.Lock() # two first requests at once must not load it twice

try:
    token = userdata.get("HF_TOKEN")
    login(token=token, add_to_git_credential=False)
    repo_files = HfApi().list_repo_files(INDICF5_REPO, token=token)
    weights = "model.safetensors" if "model.safetensors" in repo_files else \
        next(f for f in repo_files if f.endswith(".safetensors"))
    vocab = next(f for f in repo_files if f.endswith("vocab.txt"))
    indicf5_files = (hf_hub_download(INDICF5_REPO, weights, token=token),     # ~1.4 GB the first time
                     hf_hub_download(INDICF5_REPO, vocab, token=token))
    print("IndicF5 downloaded. It loads on the GPU with the first Indian-language request (~20 s).")
except Exception as e:
    indicf5_error = (f"{type(e).__name__}: {str(e)[:200]}. Check the HF_TOKEN secret (Notebook access on) "
                     f"and that you accepted the terms at https://huggingface.co/{INDICF5_REPO}")
    print("IndicF5 NOT available (XTTS and F5 still work):\n", indicf5_error)

# ---- how long the speech should be (same code as backend/indic_languages.py) ----
# ==== shared with Colab Step 3d (keep identical; tests/test_indic_languages.py checks) ====
import re, unicodedata

SYLLABLES_PER_SECOND = 5.0                # normal speaking rate (indicf5_test T7: Hindi, CER 0.00)
COMMA_PAUSE_S = 0.25                      # pause after , ; :
SENTENCE_PAUSE_S = 0.4                    # pause after । ॥ . ? ! inside the text (not after the last word)
MIN_SPEECH_S, MAX_SPEECH_S = 1.0, 20.0    # sensible limits for one sentence

# First code point of each script's Unicode block. The 9 Indian scripts share one layout
# (from the old ISCII standard), so the same offsets mean the same thing in every block.
INDIC_BLOCKS = {"Devanagari": 0x0900, "Bengali": 0x0980, "Gurmukhi": 0x0A00, "Gujarati": 0x0A80,
                "Odia": 0x0B00, "Tamil": 0x0B80, "Telugu": 0x0C00, "Kannada": 0x0C80, "Malayalam": 0x0D00}
# Scripts whose languages drop the 'a' after a bare last consonant (Hindi आप = "aap", not "aapa").
DROPS_FINAL_A = {"Devanagari", "Bengali", "Gurmukhi", "Gujarati"}
# Letters outside the common layout: Assamese ৰ ৱ, Odia ୱ (consonants); Gurmukhi ੰ ੱ (marks);
# Gurmukhi ੲ ੳ (vowel carriers); Malayalam chillus ൺ ൻ ർ ൽ ൾ ൿ (consonants with no vowel).
EXTRA_KINDS = {("Bengali", 0x70): "consonant", ("Bengali", 0x71): "consonant",
               ("Odia", 0x71): "consonant", ("Gurmukhi", 0x70): "sign", ("Gurmukhi", 0x71): "sign",
               ("Gurmukhi", 0x72): "vowel", ("Gurmukhi", 0x73): "vowel",
               **{("Malayalam", o): "dead" for o in range(0x7A, 0x80)}}


def script_of(ch):
    """Name of the Indian script a character belongs to, or None."""
    for name, start in INDIC_BLOCKS.items():
        if start <= ord(ch) < start + 0x80:
            return name
    return None


def letter_kind(ch):
    """What a character does: 'vowel' (a vowel letter, e.g. आ), 'consonant' (क), 'sign' (a vowel
    sign or nasal mark, e.g. ा ं), 'virama' (् joins two consonants), 'nukta' (the dot in ड़),
    'dead' (consonant with no vowel), 'digit', or None (anything else)."""
    if ch.isdigit():
        return "digit"
    script = script_of(ch)
    if script is None:
        return None
    o = ord(ch) - INDIC_BLOCKS[script]
    if (script, o) in EXTRA_KINDS:
        return EXTRA_KINDS[(script, o)]
    if 0x04 <= o <= 0x14 or o in (0x60, 0x61):
        return "vowel"
    if 0x15 <= o <= 0x39 or 0x58 <= o <= 0x5F:
        return "consonant"
    if o == 0x3C:
        return "nukta"
    if o == 0x4D:
        return "virama"
    if 0x01 <= o <= 0x03 or 0x3E <= o <= 0x4C or 0x55 <= o <= 0x57 or o in (0x62, 0x63):
        return "sign"
    return None


def words_of(text):
    """Words = runs of letters, marks and digits (vowel signs are marks, so they stay inside the word)."""
    return "".join(c if unicodedata.category(c)[0] in "LMN" else " " for c in text).split()


def count_syllables(text):
    """Syllables in text written in any of the 9 Indian scripts (plus any English words or digits
    mixed in). Each vowel letter is one syllable, and each consonant that carries a vowel: one
    written as a vowel sign, or the built-in 'a'. A consonant with a virama (्) joins the next
    one, so it adds nothing; in scripts whose languages drop a final 'a', a bare last consonant
    adds nothing either."""
    count = 0
    for word in words_of(text):
        chars = [c for c in word if letter_kind(c) != "nukta"]
        for i, ch in enumerate(chars):
            kind = letter_kind(ch)
            after = letter_kind(chars[i + 1]) if i + 1 < len(chars) else "end"
            if kind == "vowel":
                count += 1
            elif kind == "digit":
                count += 2                  # a digit is read as a number word, about 2 syllables
            elif kind == "consonant":
                if after == "virama":
                    continue                # part of a cluster, e.g. स्त in नमस्ते
                if after == "end" and i > 0 and script_of(ch) in DROPS_FINAL_A:
                    continue                # final 'a' dropped: मदद = "madad"
                count += 1
        latin = "".join(c for c in word if c.isascii() and c.isalpha())
        if latin:                           # an English word: one syllable per vowel group
            count += max(1, len(re.findall("[aeiouy]+", latin.lower())))
    return count


def clamp_speech(seconds):
    return round(min(max(seconds, MIN_SPEECH_S), MAX_SPEECH_S), 2)


def estimate_speech_seconds(text):
    """How long the sentence should take: syllables at a normal rate, plus short pauses at
    punctuation (none after the last word), kept between MIN_SPEECH_S and MAX_SPEECH_S."""
    body = text.strip().rstrip("।॥.?!,;: ")
    pauses = (sum(COMMA_PAUSE_S for c in body if c in ",;:")
              + sum(SENTENCE_PAUSE_S for c in body if c in "।॥.?!"))
    return clamp_speech(count_syllables(text) / SYLLABLES_PER_SECOND + pauses)


def ratio_speech_seconds(ref_seconds, ref_text, gen_text):
    """F5's own guess (reference length x text-length ratio), but counting CHARACTERS, not
    UTF-8 bytes: an Indian letter is 3 bytes and an English one 1, so F5's byte count made the
    Hindi part ~3x too long (the garbled start in indicf5_test T7). Fallback only."""
    return clamp_speech(ref_seconds * len(gen_text.strip()) / max(len(ref_text.strip()), 1))


def indicf5_total_seconds(ref_seconds, ref_text, gen_text):
    """fix_duration for IndicF5: the reference recording + the new speech. Uses the syllable
    estimate; only text with nothing countable falls back to the character ratio."""
    speech = (estimate_speech_seconds(gen_text) if count_syllables(gen_text)
              else ratio_speech_seconds(ref_seconds, ref_text, gen_text))
    return round(ref_seconds + speech, 2)
# ==== end of code shared with Colab Step 3d ====

# ---- loading (method B from indicf5_test: IndicF5's weights in f5-tts 1.1.22's own model) ----
def strip_prefix(state, prefix):
    """Weights whose names start with `prefix`, without it (and without torch.compile's '_orig_mod.')."""
    return {k.replace("_orig_mod.", "")[len(prefix):]: v for k, v in state.items()
            if k.replace("_orig_mod.", "").startswith(prefix)}

def load_indicf5():
    """Loads IndicF5 once (first request). Same model settings as f5-tts's load_model for an
    F5 v0 ("Base") model, but with IndicF5's weights, vocabulary and vocoder."""
    global indicf5
    with indicf5_lock:
        if indicf5 is not None:
            return indicf5
        if indicf5_files is None:
            raise RuntimeError(f"IndicF5 isn't set up: {indicf5_error}. Fix it, then re-run Step 3d.")
        import f5_tts.infer.utils_infer as fu
        from f5_tts.model import CFM, DiT
        from f5_tts.model.utils import get_tokenizer
        from safetensors.torch import load_file
        t0 = time.time()
        weights_path, vocab_path = indicf5_files
        vocab_char_map, vocab_size = get_tokenizer(vocab_path, "custom")
        cfm = CFM(
            transformer=DiT(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4,
                            text_mask_padding=False, pe_attn_head=1,       # v0 model (F5TTS_Base.yaml)
                            text_num_embeds=vocab_size, mel_dim=fu.n_mel_channels),
            mel_spec_kwargs=dict(n_fft=fu.n_fft, hop_length=fu.hop_length, win_length=fu.win_length,
                                 n_mel_channels=fu.n_mel_channels, target_sample_rate=fu.target_sample_rate,
                                 mel_spec_type="vocos"),
            odeint_kwargs=dict(method=fu.ode_method), vocab_char_map=vocab_char_map)
        state = load_file(weights_path, device="cpu")
        skip = {"mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"}   # rebuilt, not loaded
        result = cfm.load_state_dict({k: v for k, v in strip_prefix(state, "ema_model.").items()
                                      if k not in skip}, strict=False)
        missing = [k for k in result.missing_keys if k not in skip]
        if missing:
            raise RuntimeError(f"IndicF5 weights don't fit the model: missing {missing[:5]}")
        vocoder = fu.load_vocoder("vocos", is_local=False, device="cpu")
        if strip_prefix(state, "vocoder."):            # IndicF5 ships its own copy of the Vocos weights
            vocoder.load_state_dict(strip_prefix(state, "vocoder."), strict=False)
        del state
        indicf5 = (cfm.to(device).eval(), vocoder.to(device).eval())
        print(f"   IndicF5 loaded in {time.time() - t0:.1f}s")
        return indicf5

def indicf5_status():
    """For the API's status reply."""
    return "loaded" if indicf5 is not None else \
        "ready (loads on the first request)" if indicf5_files is not None else f"not set up: {indicf5_error}"

# ---- speaking (uses Step 4a's clip cleaning; runs only when a request arrives) ----
def f5_reference(clips_b64):
    """The same reference as F5 (longest clip, F5's trim at a pause, Whisper transcript),
    shared through F5's cache so the clip is only trimmed and transcribed once."""
    key = clips_key(clips_b64)
    if key not in f5_cache:
        clips = usable_clips(clips_b64, max_seconds=F5_MAX_REF_SECONDS)
        longest = max(clips, key=lambda c: c[1])[0]
        f5_cache[key] = f5_utils.preprocess_ref_audio_text(longest, "", show_info=lambda *a, **k: None)
        print(f"   reference: {sf.info(f5_cache[key][0]).duration:.1f}s, transcript: {f5_cache[key][1]!r}")
    return f5_cache[key]

def run_indicf5(clips_b64, req):
    """req.text must already be in the language's script (the backend converts it).
    T7 setting: English reference transcript as Whisper wrote it, an explicit duration from the
    syllable count (F5's own guess made the speech ~3x too long), 32 steps, peak-normalised."""
    cfm, vocoder = load_indicf5()
    ref_path, ref_text = f5_reference(clips_b64)
    ref_text = ref_text.strip()
    ref_s = sf.info(ref_path).duration
    total_s = indicf5_total_seconds(ref_s, ref_text, req.text)
    print(f"   IndicF5 [{req.language}]: {count_syllables(req.text)} syllables -> {total_s - ref_s:.2f}s of speech")
    wav, sr, _ = f5_utils.infer_process(ref_path, ref_text, req.text, cfm, vocoder, mel_spec_type="vocos",
                                        speed=1.0, nfe_step=INDICF5_NFE_STEPS, fix_duration=total_s,
                                        device=device, show_info=lambda *a, **k: None, progress=None)
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    peak = float(np.max(np.abs(wav)))
    if peak > 0:
        wav = wav * (10 ** (INDICF5_PEAK_DBFS / 20) / peak)
    return wav, sr, {"ref_text": ref_text, "speech_seconds_planned": round(total_s - ref_s, 2)}

print("IndicF5 status:", indicf5_status())

## Step 4 — Quick test of both models

In [ ]:
from IPython.display import Audio, display
from importlib.resources import files

print("XTTS-v2:")
display(Audio(tts.tts(text="Hello, this is the XTTS voice.", speaker="Ana Florence", language="en"), rate=SAMPLE_RATE))

if f5 is not None:
    print("F5-TTS:")
    ref = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
    wav, sr, _ = f5.infer(ref_file=ref,
                          ref_text="Some call me nature, others call me mother nature.",
                          gen_text="Hello, this is the F5 voice.")
    display(Audio(wav, rate=sr))

## Step 4a — Load the ReVoice voice functions + metrics
The code that turns text + reference recordings into speech, and measures each result (similarity, pitch, speaking rate, accent). The API in Step 6 and the test in Step 4b both use it.

In [ ]:
# Voice functions shared by the Step 4b test and the Step 6 API, so the test runs the same code.
import os, base64, hashlib, subprocess, tempfile, time
import soundfile as sf
from types import SimpleNamespace
try:
    import f5_tts.infer.utils_infer as f5_utils     # F5's own reference trimming + caches
except Exception:
    f5_utils = None                                 # F5 not installed: XTTS still works

xtts_cache = {}   # voice profiles for XTTS
f5_cache = {}     # (trimmed reference clip, its transcript) for F5
temp_files = []   # cleaned clips on disk, deleted by forget_all_voices()

# XTTS: emotion changes speaking speed and how "lively" it sounds
EMOTION_STYLE = {
    "neutral": {"speed": 1.00, "temperature": 0.80},
    "happy":   {"speed": 1.05, "temperature": 0.85},
    "urgent":  {"speed": 1.12, "temperature": 0.85},
    "tender":  {"speed": 0.92, "temperature": 0.75},
}

# F5 settings
F5_SPEED = 1.0              # always normal speed: the reference recording carries the emotion
F5_NFE_STEPS = 64           # more refinement steps = clearer speech (32 was a bit rough)
F5_MAX_REF_SECONDS = 15     # pass up to 15s; F5 then trims to <=12s at a pause by itself

def clean_clip(b64, max_seconds=None):
    """Decode a browser recording and clean it for cloning."""
    raw = base64.b64decode(b64)
    f = tempfile.NamedTemporaryFile(suffix=".input", delete=False)
    f.write(raw); f.close()
    out = f.name + ".wav"
    cmd = ["ffmpeg", "-y", "-i", f.name, "-ac", "1", "-ar", "24000",
           "-af", "highpass=f=70,"
                  "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,"
                  "loudnorm=I=-18:TP=-2:LRA=11"]
    if max_seconds:
        cmd += ["-t", str(max_seconds)]
    r = subprocess.run(cmd + [out], capture_output=True, text=True)
    os.remove(f.name)              # the raw upload isn't needed any more
    if r.returncode != 0:
        raise RuntimeError("ffmpeg failed:\n" + r.stderr[-1200:])
    temp_files.append(out)
    return out

def usable_clips(clips_b64, max_seconds=None):
    out = []
    for b64 in clips_b64:
        p = clean_clip(b64, max_seconds)
        d = sf.info(p).duration
        if d >= 0.8:
            out.append((p, d))
    if not out:
        raise RuntimeError("All reference clips were too short or silent. Re-record and keep talking.")
    return out

def clips_key(clips_b64):
    return hashlib.md5("".join(clips_b64).encode()).hexdigest()

# XTTS cloned from the user's recordings
def run_xtts(clips_b64, req, style):
    key = clips_key(clips_b64)
    if key not in xtts_cache:
        clips = usable_clips(clips_b64)
        print(f"   XTTS profile from {sum(d for _, d in clips):.1f}s of audio")
        xtts_cache[key] = xtts_model.get_conditioning_latents(
            audio_path=[p for p, _ in clips], gpt_cond_len=30, gpt_cond_chunk_len=4, max_ref_length=30)
    gpt_cond_latent, speaker_embedding = xtts_cache[key]
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# XTTS stock (built-in) voice
def stock_speaker_names():
    return sorted(xtts_model.speaker_manager.speaker_names)

def run_xtts_stock(req, style):
    speakers = xtts_model.speaker_manager.speakers
    if req.stock_speaker not in speakers:
        raise RuntimeError(f"Unknown stock voice {req.stock_speaker!r}. "
                           f"Some valid ones: {', '.join(stock_speaker_names()[:8])}")
    # built-in voices come with their voice profile already computed
    gpt_cond_latent, speaker_embedding = speakers[req.stock_speaker].values()
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# F5-TTS
def run_f5(clips_b64, req):
    if f5 is None:
        raise RuntimeError("F5-TTS isn't loaded. Check the output of Step 3b.")
    key = clips_key(clips_b64)
    if key not in f5_cache:
        # F5 uses ONE reference: the longest clip, up to 15 seconds
        clips = usable_clips(clips_b64, max_seconds=F5_MAX_REF_SECONDS)
        longest = max(clips, key=lambda c: c[1])[0]
        # F5 trims first (<=12 s, at a pause); the empty transcript makes Whisper transcribe the trimmed
        # audio, so the words match what F5 hears (a mismatch causes mumbled or skipped words).
        ref_path, ref_text = f5_utils.preprocess_ref_audio_text(longest, "", show_info=lambda *a, **k: None)
        print(f"   F5 reference: {sf.info(ref_path).duration:.1f}s, transcript: {ref_text!r}")
        f5_cache[key] = (ref_path, ref_text)
    ref_path, ref_text = f5_cache[key]
    wav, sr, _ = f5.infer(ref_file=ref_path, ref_text=ref_text, gen_text=req.text,
                          speed=F5_SPEED, nfe_step=F5_NFE_STEPS, show_info=lambda *a, **k: None)
    return wav, sr, {"ref_text": ref_text.strip()}

# Voice metrics for the reference and the output: speaker similarity (ECAPA cosine, 1 = same voice),
# pitch, speaking rate, and accent as top-3 raw scores (not probabilities). A failed metric is None.
import numpy as np, librosa, torch

METRICS_ENABLED = True       # set False to skip metrics (makes each request ~1-3 s faster)
METRIC_SR = 16000            # both SpeechBrain models expect 16 kHz audio
reference_metrics_cache = {}

# keep models loaded in Step 3c (None if that step was skipped or failed)
spk_model = globals().get("spk_model")
accent_model = globals().get("accent_model")

def safe(name, fn, *args):
    """Runs one metric; on any error prints it and returns None instead of crashing."""
    try:
        return fn(*args)
    except Exception as e:
        print(f"   metric '{name}' failed: {type(e).__name__}: {e}")
        return None

def speaker_embedding(wav16):
    with torch.no_grad():
        return spk_model.encode_batch(torch.tensor(wav16).float()).squeeze().cpu()

def pitch_stats(wav16):
    f0, voiced, _ = librosa.pyin(wav16, fmin=65, fmax=400, sr=METRIC_SR, frame_length=1024)
    f0 = f0[voiced & ~np.isnan(f0)]
    if len(f0) < 10:
        return None, None
    return round(float(np.mean(f0)), 1), round(float(np.percentile(f0, 95) - np.percentile(f0, 5)), 1)

def speech_seconds_of(wav16):
    """Seconds of actual speech (pauses removed)."""
    parts = librosa.effects.split(wav16, top_db=35)
    return sum(e - s for s, e in parts) / METRIC_SR

def words_per_second(wav16, n_words):
    if not n_words:
        return None
    speech_seconds = speech_seconds_of(wav16)
    return round(n_words / speech_seconds, 2) if speech_seconds > 0.3 else None

def accent_scores(wav16):
    """The 3 most likely accents with their RAW scores: [[label, score], ...], best first.
    This model's outputs are cosine scores (-1..1), NOT probabilities, so we pass them on
    as they are. (Before v8 they went through x30 + softmax: an unconfirmed scale that
    showed e.g. "US 100%" when US was only 0.2 ahead.)"""
    with torch.no_grad():
        out, _, _, _ = accent_model.classify_batch(torch.tensor(wav16).float().unsqueeze(0))
    scores = out.squeeze(0).float().cpu()
    labels = accent_model.hparams.label_encoder.ind2lab
    best = torch.argsort(scores, descending=True)[:3]
    return [[labels[int(i)], round(float(scores[i]), 3)] for i in best]

def voice_stats(wav16, n_words):
    """Pitch, rate and accent for one piece of audio (each None if it can't be measured)."""
    pitch = safe("pitch", pitch_stats, wav16) or (None, None)
    return {"pitch_hz": pitch[0], "pitch_range_hz": pitch[1],
            "words_per_sec": safe("rate", words_per_second, wav16, n_words),
            "speech_seconds": safe("speech", lambda: round(speech_seconds_of(wav16), 2)),
            "accent_scores": safe("accent", accent_scores, wav16) if accent_model is not None else None}

def reference_metrics(clips_b64):
    """Measures the real voice once per set of clips (cached): up to 30 s of the cleaned clips."""
    key = clips_key(clips_b64)
    if key not in reference_metrics_cache:
        clips = usable_clips(clips_b64)
        wav16 = np.concatenate([librosa.load(p, sr=METRIC_SR, mono=True)[0] for p, _ in clips])[:30 * METRIC_SR]
        # words in the reference: Whisper writes down what was said (needs F5's Whisper)
        n_words = None
        if f5 is not None:
            text = safe("transcribe", lambda: " ".join(f5.transcribe(p) for p, _ in clips))
            n_words = len(text.split()) if text else None
        reference_metrics_cache[key] = {
            "stats": voice_stats(wav16, n_words),
            "embedding": safe("embedding", speaker_embedding, wav16) if spk_model is not None else None,
        }
    return reference_metrics_cache[key]

def compute_metrics(req, clips_b64, wav, sr):
    """All metrics for one generated sentence. Stock voices have no reference recording,
    so only the output is measured for them."""
    out16 = librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=sr, target_sr=METRIC_SR)
    metrics = {"similarity": None, "reference": None, "accent_match": None,
               "output": voice_stats(out16, len(req.text.split()))}
    if clips_b64:
        ref = safe("reference", reference_metrics, clips_b64)
        if ref:
            metrics["reference"] = ref["stats"]
            if ref["embedding"] is not None:
                out_emb = safe("embedding", speaker_embedding, out16)
                if out_emb is not None:
                    metrics["similarity"] = round(float(
                        torch.nn.functional.cosine_similarity(ref["embedding"], out_emb, dim=0)), 3)
            # accent_match is left to the backend/page (needs enough speech and a clear winner on both sides)
    return metrics

# Single entry point, used by the API and by Step 4b
def generate(req, clips):
    """Makes the speech. Returns (wav, sample_rate, extra_info)."""
    style = EMOTION_STYLE.get(req.emotion, EMOTION_STYLE["neutral"])
    if req.stock_speaker:
        # stock voice: no recordings needed (XTTS only)
        if req.engine != "xtts":
            raise RuntimeError("Stock voices only work with XTTS-v2.")
        print(f"\n--- [xtts stock] {req.text!r} | {req.emotion} | {req.stock_speaker}")
        return run_xtts_stock(req, style)
    if not clips:
        raise RuntimeError("No reference audio sent.")
    print(f"\n--- [{req.engine}] {req.text!r} | {req.emotion} | {len(clips)} clip(s)")
    if req.engine == "f5":
        return run_f5(clips, req)
    if req.engine == "xtts":
        return run_xtts(clips, req, style)
    if req.engine == "indicf5":
        if "run_indicf5" not in globals():
            raise RuntimeError("IndicF5 isn't set up: run Step 3d, then Steps 4a and 6 again.")
        return run_indicf5(clips, req)          # Step 3d
    raise RuntimeError(f"Unknown engine {req.engine!r}.")

def synthesize_core(req):
    """Returns (wav, sample_rate, extra_info). req needs: text, language, emotion,
    engine, stock_speaker, speaker_wav_base64, speaker_wav_base64_list.
    extra_info has synth_seconds, metrics, metrics_seconds (and ref_text for F5)."""
    clips = [] if req.stock_speaker else (
        req.speaker_wav_base64_list or ([req.speaker_wav_base64] if req.speaker_wav_base64 else []))
    t0 = time.time()
    wav, sr, extra = generate(req, clips)
    extra["synth_seconds"] = round(time.time() - t0, 2)      # speech generation only

    extra["metrics"] = None
    if METRICS_ENABLED:
        t1 = time.time()
        extra["metrics"] = safe("metrics", compute_metrics, req, clips, wav, sr)
        extra["metrics_seconds"] = round(time.time() - t1, 2)
        print(f"   metrics: {extra['metrics']}")
    return wav, sr, extra

def forget_all_voices():
    """Wipes every cached voice and cleaned clip (called when a user is deleted)."""
    n = len(xtts_cache) + len(f5_cache)
    xtts_cache.clear(); f5_cache.clear()
    # F5 keeps its own cache of trimmed clips and transcripts: wipe that too
    if f5_utils is not None:
        temp_files.extend(f5_utils._ref_audio_cache.values())
        f5_utils._ref_audio_cache.clear(); f5_utils._ref_text_cache.clear()
    for path in temp_files:
        try:
            os.remove(path)
        except OSError:
            pass
    temp_files.clear()
    print(f"\n--- forgot {n} cached voice(s)")
    return n

print("Voice functions ready.")

## Step 4b — Test with YOUR voice (optional, for troubleshooting)
Upload a short recording (6–20 s). XTTS (and F5) speak a fixed sentence through the exact API code.
- Gibberish **here too** → environment problem: check Step 2b, redo Step 2.
- Fine here but bad in the app → the reference clips sent by the app are the problem (re-record).

In [ ]:
# Upload a recording; XTTS (and F5) speak a fixed sentence through the exact API code.
# Gibberish here too = environment problem (redo Step 2); fine here but bad in the app = the app's clips.
import time
from google.colab import files as colab_files
from IPython.display import Audio, display

TEST_SENTENCE = "Hello, this is a quick test of my cloned voice. Does it sound like me?"
ALSO_TEST_F5 = True     # set to False to test XTTS only

uploaded = colab_files.upload()
for name, data in uploaded.items():
    req = SimpleNamespace(text=TEST_SENTENCE, language="en", emotion="neutral", engine="xtts",
                          stock_speaker=None, speaker_wav_base64=None,
                          speaker_wav_base64_list=[base64.b64encode(data).decode()])
    t0 = time.time()
    wav, sr, info = synthesize_core(req)     # info = extra details, incl. metrics
    print(f"{name} -> XTTS ({time.time() - t0:.1f}s):")
    display(Audio(wav, rate=sr))
    print("   metrics:", info["metrics"])

    if ALSO_TEST_F5 and f5 is not None:
        req.engine = "f5"
        t0 = time.time()
        wav, sr, extra = synthesize_core(req)
        print(f"{name} -> F5 ({time.time() - t0:.1f}s). Whisper heard the reference as: {extra['ref_text']!r}")
        display(Audio(wav, rate=sr))
        print("   metrics:", extra["metrics"])

## Step 5 — ngrok token
https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

## Step 6 — Start the API (both engines)
Safe to re-run any time. Your ngrok URL stays the same.
The last line must say **ReVoice API v9** and lists which engines are loaded.
Needs Step 4a to have been run (the API uses the same voice functions as Step 4b).

In [ ]:
import traceback, threading, time, requests, socket, io
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
from pyngrok import ngrok

# stop a server/tunnel left over from running this cell before
try:
    server.should_exit = True; time.sleep(2)
except NameError:
    pass
try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

app = FastAPI()

class SynthRequest(BaseModel):
    text: str
    speaker_wav_base64: Optional[str] = None
    speaker_wav_base64_list: Optional[List[str]] = None
    language: str = "en"
    emotion: str = "neutral"
    engine: str = "xtts"
    stock_speaker: Optional[str] = None   # built-in XTTS voice, for users with no recordings

@app.get("/")
def health():
    # v7 = v6 + voice evaluation metrics in every /synthesize reply
    # v8 = accent as top-3 raw scores + seconds of speech (no percentages)
    # v9 = engine "indicf5" (Step 3d: 11 Indian languages) + which engines are loaded
    indic = globals().get("indicf5_status", lambda: "not set up: Step 3d wasn't run")()
    return {"status": "ReVoice API v9 running", "xtts": True, "f5": f5 is not None,
            "indicf5": indic, "engines": ["xtts"] + (["f5"] if f5 is not None else [])
                                         + (["indicf5"] if not indic.startswith("not set up") else [])}

@app.get("/speakers")
def speakers():
    """Names of XTTS's built-in voices (for the 'use a stock voice' option)."""
    return {"speakers": stock_speaker_names()}

@app.post("/forget_voices")
def forget_voices():
    """Wipes every cached voice profile and cleaned clip (called when a user is deleted)."""
    return {"forgotten": forget_all_voices()}

@app.post("/synthesize")
def synthesize(req: SynthRequest):
    try:
        t0 = time.time()
        wav, sr, extra = synthesize_core(req)        # defined in Step 4a
        print(f"   generated in {time.time() - t0:.1f}s")
        buf = io.BytesIO()
        sf.write(buf, wav, samplerate=sr, format="WAV")
        # extra: synth_seconds, metrics, metrics_seconds, and ref_text for F5
        return {"audio_base64": base64.b64encode(buf.getvalue()).decode("utf-8"), **extra}
    except Exception as e:
        print(traceback.format_exc())
        return JSONResponse({"error": f"{type(e).__name__}: {e}"}, status_code=500)

def free_port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p

PORT = free_port()
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

# wait (up to 30s) until the server is really listening, instead of guessing
for _ in range(60):
    if server.started:
        break
    time.sleep(0.5)
else:
    raise RuntimeError("The API server did not start within 30 seconds. "
                       "Look for an error message above, then re-run this cell.")

tunnel = ngrok.connect(PORT)
print("=" * 65)
print("URL for COLAB_API_URL in backend/.env:")
print(tunnel.public_url)
print("=" * 65)

# health check through ngrok; show the raw reply if it isn't JSON (e.g. an ngrok error page)
reply = requests.get(tunnel.public_url, headers={"ngrok-skip-browser-warning": "true"}, timeout=30)
try:
    print(reply.json())
except ValueError:
    print(f"Health check did not return JSON (HTTP {reply.status_code}). Raw reply:")
    print(reply.text[:500])

## Keep this tab open
If Colab disconnects: run Steps 3, 3b, 3c, 3d, 4a, 5 and 6 again (Step 2 only if packages are missing).
Never run cells from older notebooks — the URL always points to whichever server started last.